# 1. Imports & Setup

In [ ]:
import re
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import truncnorm, norm, mannwhitneyu, ks_2samp
from scipy.linalg import cholesky

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             accuracy_score, f1_score, precision_score, recall_score)
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 2. Load Source Tables

In [ ]:
data_paths = {
    1: '/content/drive/MyDrive/data_1 - HTML Table to Markdown Conversion.csv',
    2: '/content/drive/MyDrive/data_2 - HTML Table to Markdown Conversion.csv',
    3: '/content/drive/MyDrive/data_3 - HTML to Markdown Table Conversion (2).csv',
    4: '/content/drive/MyDrive/data_4 - HTML Table to Markdown Conversion.csv',
    5: '/content/drive/MyDrive/data_5 - Table Data Conversion to Markdown.csv',
    6: '/content/drive/MyDrive/data_6 - Fatty Acid Composition Comparison Table.csv',
}

df_dict = {}
for i, path in data_paths.items():
    df_dict[i] = pd.read_csv(path)
    print(f"Loaded df_dict[{i}] from {path.split('/')[-1]}  shape={df_dict[i].shape}")


In [ ]:
# Quick look at the first table
df_dict[1].head()


# 3. EDA: Column Parsing & Cleaning

In [ ]:
def divide_by_symbol(symb, df, target_column, new_col_1, new_col_2):
    split_data = df[target_column].astype(str).str.split(symb, expand=True)
    df.drop(columns=[target_column], inplace=True)
    if split_data.shape[1] == 2:
        df[new_col_1] = split_data[0].str.strip()
        df[new_col_2] = split_data[1].str.strip()
    else:
        print(f"{symb} is not in  {target_column}")

    return df

import re

def clean_scientific_notation(text):
    if not isinstance(text, str):
        return text

    # 1. Удаляем HTML теги <sup> и </sup>
    text = re.sub(r'<sup>', '', text)
    text = re.sub(r'</sup>', '', text)

    # 2. Заменяем символ умножения '×' и '10' на 'e'
    # Например: "4.95 × 10-9" станет "4.95e-9"
    text = re.sub(r'\s*×\s*10', 'e', text)

    # 3. Убираем лишние пробелы и спецсимволы минуса (иногда бывает длинное тире –)
    text = text.replace('−', '-').replace(' ', '')

    try:
        return float(text)
    except:
        return text # Если это не число (например, заголовок), оставляем как есть


def auto_clean_values(val):
    if not isinstance(val, str):
        return val

    # 1. Удаляем HTML теги типа <sup>...</sup>
    clean_val = re.sub(r'<[^>]+>', '', val)

    # 2. Обработка научной нотации (например, 4.95 × 10−9)
    # Заменяем символ умножения и "10" на "e", исправляем минус
    clean_val = clean_val.replace('×', 'e').replace('10', '').replace('−', '-')
    clean_val = re.sub(r'\s+', '', clean_val) # удаляем пробелы

    # 3. Если после очистки это выглядит как научное число (напр. 4.95e-9)
    if re.search(r'\d+e-?\d+', clean_val):
        try:
            return float(clean_val)
        except:
            pass

    # 4. Обработка обычных чисел и возвращение результата
    try:
        return float(clean_val)
    except:
        return clean_val

In [ ]:
sample_number_dict={}
pattern = r"n=(\d+)"
for i in range(1,7):
  columns=df_dict[i].columns
  for column in columns:
    match = re.search(pattern, column)
    if match:
        number = match.group(1)
        print(f"Detected number: {number}--> dict--> {i} ---> {column}")
        sample_number_dict[i]=sample_number_dict.get(i,set())
        sample_number_dict[i].add(int(number))



In [ ]:
for i in range(1, 7):
    print(f'__________________ Processing DataFrame: {i}')
    df = df_dict[i]
    columns = df.columns
    columns_to_change = [col for col in columns if '±' in str(col)]

    for column in columns_to_change:
        try:
            parts = column.split(':')
            main_name = parts[0].strip()
            if len(parts) > 1 and '±' in parts[1]:
                sub_parts = parts[1].split('±')
                new_col_name_1 = f"{main_name} {sub_parts[0].strip()}"
                new_col_name_2 = f"{main_name} {sub_parts[1].strip()}"

                df = divide_by_symbol('±', df, column, new_col_name_1, new_col_name_2)
            else:
                sub_parts = column.split('±')
                df = divide_by_symbol('±', df, column, f"{column}_mean", f"{column}_std")

            df_dict[i] = df
        except Exception as e:
            print(f"error {column}: {e}")

In [ ]:
# ПРИМЕНЕНИЕ КО ВСЕМ ДАННЫМ:
for i in range(1, 7):
    # Применяем функцию к каждой ячейке каждого столбца
    df_dict[i] = df_dict[i].applymap(auto_clean_values)
    # Пытаемся принудительно конвертировать столбцы в числа, если это возможно
    df_dict[i] = df_dict[i].apply(pd.to_numeric, errors='ignore')
    print(f"Таблица {i} автоматически очищена и сконвертирована.")
# Проверка результата
print(df_dict[1].head())


In [ ]:
N = 50  # fish per group

# ============================================================
# 1. DATA CLEANING — məlum xətaları düzəlt
# ============================================================
def clean_data(df_dict):
    """Cədvəllərdəki yazılış xətalarını düzəldir."""
    # Cədvəl 1: Bodymaximumheight AG Mean = 0.01 (səhv) → 10.65
    bio = df_dict[1].copy()
    mask = bio["Biometric Traits"] == "Bodymaximumheight(cm)"
    bio.loc[mask, "AG (n=50) Mean"] = 10.65  # (9.5+11.8)/2
    df_dict[1] = bio

    # Cədvəl 6: C20:1n-9 SEM = "1.23e-" (korrupsiya) → 1.23e-05
    fa = df_dict[6].copy()
    fa["AG (n=6) SEM"] = pd.to_numeric(fa["AG (n=6) SEM"], errors="coerce")
    fa["RG (n=6) SEM"] = pd.to_numeric(fa["RG (n=6) SEM"], errors="coerce")
    mask = fa["Fatty Acids"].astype(str).str.strip() == "C20:1n-9"
    fa.loc[mask, "AG (n=6) SEM"] = 1.23e-05
    df_dict[6] = fa

    return df_dict

df_dict = clean_data(df_dict)

# 4. Synthetic Data Helpers (Truncated Normal & Gaussian Copula)

In [ ]:
def sem_to_sd(sem, n):
    """
    Elmi əsas: SEM = SD / √n  →  SD = SEM × √n
    Bu, statistikanın təməl tənliyidir (Altman & Bland 2005, BMJ 331:903).
    """
    return sem * np.sqrt(n)

In [ ]:
def sample_truncated(mean, sd, lo, hi, n, rng):
    """
    Elmi əsas: Real balıq çəkisi mənfi ola bilməz, FA faizi 100-dən
    böyük ola bilməz. Truncated normal paylanma bu hədləri qoruyur
    (Burkardt 2014, Johnson & Kotz 1994).

    mean, sd: arzu olunan mean və SD
    lo, hi:   fiziki aralıq (cədvəldən Min/Max və ya 0)
    """
    if sd <= 0:
        return np.full(n, mean)

    # Əgər mean aralıqdan kənardırsa (cədvəl xətası), düzəlt
    if not (pd.isna(lo) or pd.isna(hi)) and (mean < lo or mean > hi):
        mean = (lo + hi) / 2

    if pd.isna(lo): lo = max(0, mean - 6*sd)
    if pd.isna(hi): hi = mean + 6*sd

    a, b = (lo - mean) / sd, (hi - mean) / sd
    seed_int = int(rng.integers(1, 2**31 - 1))
    return truncnorm.rvs(a, b, loc=mean, scale=sd, size=n, random_state=seed_int)

In [ ]:
def gaussian_copula_sample(means, sds, los, his, corr_matrix, n, rng):
    """
    Elmi əsas: Balıq morfometriyasında body_mass, total_length, head_length
    yüksək korrelyasiyalıdır (r≈0.95, Clarias gariepinus: Udoh & Ukpatu 2017).
    Müstəqil sample etmək bu strukturu pozur.

    Gaussian copula: korrelyasiyalı MVN → uniform → truncated normal
    (Nelsen 2006; Cario & Nelson 1997, NORTA).
    """
    k = len(means)
    # Cholesky ilə korrelyasiyalı standart normal
    L = np.linalg.cholesky(corr_matrix)
    z = rng.standard_normal((n, k)) @ L.T

    # Hər ölçü üçün: Z → uniform → truncated normal
    out = np.zeros_like(z)
    for j in range(k):
        u = norm.cdf(z[:, j])  # standard normal → uniform [0,1]
        lo, hi = los[j], his[j]
        mean, sd = means[j], sds[j]

        if mean < lo or mean > hi:
            mean = (lo + hi) / 2

        a, b = (lo - mean) / sd, (hi - mean) / sd
        out[:, j] = truncnorm.ppf(u, a, b, loc=mean, scale=sd)

    return out

# 5. Synthetic Data Generation

In [ ]:
for i in range(1, 7):
  display(df_dict[i])
  #1 , 2 ,3, 6 are non time based
  #4, 5 time based


In [ ]:
non_time_based=[1,2,3,6]
time_based=[5,4]

In [ ]:
df_dict[1] = df_dict[1][df_dict[1]['Biometric Traits'] == 'Bodymass(g)']

In [ ]:
"""
Gaussian Copula Synthetic Data Generator for Silurus glanis
===========================================================

Correlation matrices from peer-reviewed S. glanis literature:
    Bergström et al. (2022) Sci. Rep. 12:8070
    Yazıcı & Yazıcıoğlu (2020) J. Anat. Environ. Anim. Sci. 5(2):199–204
    Simeanu et al. (2022) Agriculture 12(12):2144
    Ljubojević et al. (2013) Czech J. Food Sci. 31(5):445–450
    Jankowska et al. (2007) Eur. Food Res. Technol. 224:453–459
    Hallier et al. (2007) Food Chem. 103:808–815

Usage:
    synthetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42)
"""




# ══════════════════════════════════════════════════════════════════════════
# 1. AUTO-FIX COLUMN SHIFT
# ══════════════════════════════════════════════════════════════════════════

def fix_index_shift(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect and fix DataFrame where an extra numeric index column
    has shifted all data one column to the right.

    Skip if first column header contains meaningful keywords.
    """
    df = df.copy()
    col0 = str(df.columns[0]).lower()

    # Don't drop if header name suggests real data
    skip_keywords = ['storage', 'days', 'interval', 'period', 'time',
                     'fatty', 'acid', 'group']
    if any(kw in col0 for kw in skip_keywords):
        return df

    vals = df.iloc[:, 0].dropna()
    all_numeric = True
    for v in vals:
        try:
            int(float(v))
        except (ValueError, TypeError):
            if str(v).lower() not in ('p-value', 'nan', ''):
                all_numeric = False
                break

    if all_numeric and len(vals) > 0:
        # Check: is the SECOND column all strings (feature names)?
        col1_vals = df.iloc[:, 1].dropna()
        all_str = all(isinstance(v, str) and not v.replace('.','').replace('-','').isdigit()
                      for v in col1_vals.head(5))
        if all_str:
            df = df.iloc[:, 1:].reset_index(drop=True)

    return df


def to_float(val: Any) -> Optional[float]:
    """Convert any value to float, handling dates, ND, timestamps."""
    if val is None:
        return np.nan
    if isinstance(val, (int, float, np.integer, np.floating)):
        v = float(val)
        return v if np.isfinite(v) else np.nan
    s = str(val).strip()
    if s in ('', '-', 'ND', 'nd', 'NaN', 'nan', 'None', 'NaT', '>0.9999'):
        return np.nan
    try:
        return float(s)
    except ValueError:
        pass
    if hasattr(val, 'year'):
        y, m, d = val.year, val.month, val.day
        if y > 2030:
            return float(f"{y}.{m:02d}")
        elif 2025 <= y <= 2030:
            return float(f"{d}.0{m}") if m < 10 else float(f"{d}.{m}")
    dm = re.match(r'(\d{4})-(\d{2})-(\d{2})', s)
    if dm:
        y, m, d = int(dm.group(1)), int(dm.group(2)), int(dm.group(3))
        if y > 2030:
            return float(f"{y}.{m:02d}")
        elif y >= 2025:
            return float(f"{d}.0{m}") if m < 10 else float(f"{d}.{m}")
    return np.nan


def find_cols(columns, group, keyword):
    """Find columns for a group matching keyword, case-insensitive."""
    return [c for c in columns if group in c and keyword.lower() in c.lower()]


def parse_mean_sem_str(val):
    s = str(val).strip()
    if '±' in s:
        parts = s.split('±')
        try:
            return float(parts[0].strip()), float(parts[1].strip())
        except (ValueError, IndexError):
            pass
    return None, None


# ══════════════════════════════════════════════════════════════════════════
# 2. PARSERS
# ══════════════════════════════════════════════════════════════════════════

def parse_type_A(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Tables 1,2,3: auto-detect separate vs combined Mean/SEM."""
    df = fix_index_shift(df)
    feature_col = df.columns[0]
    result = {}

    for group in ['RG', 'AG']:
        gcols = [c for c in df.columns if group in c]

        # Find Mean and SEM columns
        mean_cols = find_cols(df.columns, group, 'Mean')
        sem_cols = find_cols(df.columns, group, 'SEM')
        min_cols = find_cols(df.columns, group, 'Min')
        max_cols = find_cols(df.columns, group, 'Max')

        # Filter: combined has both Mean and SEM in same column name
        combined = [c for c in mean_cols if 'SEM' in c or '±' in c]
        mean_only = [c for c in mean_cols if c not in combined and 'SEM' not in c]
        sem_only = [c for c in sem_cols if c not in combined and 'Mean' not in c]

        if mean_only and sem_only:
            mc, sc, use_combined = mean_only[0], sem_only[0], False
        elif combined:
            mc, sc, use_combined = combined[0], None, True
        else:
            continue

        n_match = re.search(r'n=(\d+)', mc)
        n_orig = int(n_match.group(1)) if n_match else 50
        min_c = min_cols[0] if min_cols else None
        max_c = max_cols[0] if max_cols else None

        rows = []
        for _, row in df.iterrows():
            feat = str(row[feature_col]).strip()
            # Skip if feature is a number (stale index)
            try:
                float(feat)
                continue
            except ValueError:
                pass

            if use_combined:
                mean, sem = parse_mean_sem_str(row[mc])
            else:
                mean = to_float(row[mc])
                sem = to_float(row[sc])

            if mean is None or (isinstance(mean, float) and np.isnan(mean)):
                continue
            if sem is None or (isinstance(sem, float) and np.isnan(sem)):
                sem = 0.0

            sd = sem * np.sqrt(n_orig)
            mn = to_float(row[min_c]) if min_c else None
            mx = to_float(row[max_c]) if max_c else None
            if mn is None or np.isnan(mn):
                mn = mean - 3 * sd if sd > 0 else mean * 0.8
            if mx is None or np.isnan(mx):
                mx = mean + 3 * sd if sd > 0 else mean * 1.2
            # Safety: ensure min < mean < max
            mn = min(mn, mean)
            mx = max(mx, mean)
            if mn >= mx:
                mn, mx = mean * 0.9, mean * 1.1

            rows.append({
                'feature': feat, 'mean': mean, 'sem': sem,
                'sd': max(sd, 1e-6), 'min': mn, 'max': mx, 'n': n_orig
            })

        if rows:
            result[group] = pd.DataFrame(rows)

    return result


def parse_type_B(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Tables 4,5: time-series with separate Mean/SEM columns."""
    df = fix_index_shift(df)
    time_col = df.columns[0]
    group_col = None
    n_col = None

    # Find group and n columns
    for c in df.columns:
        cl = c.lower()
        if 'group' in cl:
            group_col = c
        elif cl == 'n':
            n_col = c

    if group_col is None:
        # Try column index 2
        group_col = df.columns[2] if len(df.columns) > 2 else None
    if n_col is None:
        n_col = df.columns[1] if len(df.columns) > 1 else None

    # Find measure columns (everything that's not time/n/group)
    skip = {time_col, n_col, group_col}
    measure_info = []  # list of (base_name, mean_col, sem_col)

    remaining = [c for c in df.columns if c not in skip]

    # Pair Mean/SEM columns
    used = set()
    for c in remaining:
        if c in used:
            continue
        cl = c.lower()
        if 'mean' in cl:
            base = c.split('Mean')[0].strip().rstrip(':').strip()
            # Find matching SEM
            sem_c = None
            for c2 in remaining:
                if c2 != c and c2 not in used and 'sem' in c2.lower():
                    base2 = c2.split('SEM')[0].strip().rstrip(':').strip()
                    if base2 == base or base.lower() in c2.lower():
                        sem_c = c2
                        break
            measure_info.append((base, c, sem_c))
            used.add(c)
            if sem_c:
                used.add(sem_c)
        elif '±' in cl or ('mean' in cl and 'sem' in cl):
            base = c.split(':')[0].strip()
            measure_info.append((base, c, None))
            used.add(c)

    # If no Mean/SEM pattern found, treat remaining as combined columns
    if not measure_info:
        for c in remaining:
            if c not in used:
                base = c.split(':')[0].strip()
                measure_info.append((base, c, None))

    result = {}
    for _, row in df.iterrows():
        t = str(row[time_col]).strip()
        g = str(row[group_col]).strip() if group_col else ''
        if t.lower() == 'p-value' or g in ('NaN', 'nan', '', 'None'):
            continue
        try:
            day = int(float(t))
        except (ValueError, TypeError):
            continue

        nv = to_float(row[n_col]) if n_col else 6
        n_orig = int(nv) if nv and not np.isnan(nv) else 6

        for base, mc, sc in measure_info:
            if sc:
                mean = to_float(row[mc])
                sem = to_float(row[sc])
            else:
                mean, sem = parse_mean_sem_str(row[mc])
                if mean is None:
                    mean = to_float(row[mc])
                    sem = 0.0

            if mean is None or np.isnan(mean):
                continue
            if sem is None or np.isnan(sem):
                sem = 0.0

            sd = sem * np.sqrt(n_orig) if sem > 0 else 0.01
            entry = {
                'feature': f"{base}_day{day}", 'feature_base': base,
                'mean': mean, 'sem': sem, 'sd': sd,
                'min': max(0, mean - 3 * sd),
                'max': min(100, mean + 3 * sd),
                'n': n_orig, 'day': day
            }
            result.setdefault(g, []).append(entry)

    return {k: pd.DataFrame(v) for k, v in result.items()}


def parse_type_C(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Table 6: Fatty Acids with separate or combined Mean/SEM."""
    df = fix_index_shift(df)
    fa_col = df.columns[0]

    summary_names = {
        'ΣSFA', 'ΣMUFA', 'ΣPUFA', 'Σ SFA', 'Σ MUFA', 'Σ PUFA',
        'n-3', 'n-6', 'n−3', 'n−6',
        'n-3/n-6', 'n-6/n-3', 'n−3/n−6', 'n−6/n−3',
        'PUFA/SFA', 'USFA/SFA', 'PI', 'AI', 'TI',
        'HFA', 'hFA', 'h/H'
    }
    result = {}

    for group in ['AG', 'RG']:
        mean_cols = find_cols(df.columns, group, 'Mean')
        sem_cols = find_cols(df.columns, group, 'SEM')
        combined = [c for c in mean_cols if 'SEM' in c or '±' in c]
        mean_only = [c for c in mean_cols if c not in combined and 'SEM' not in c]
        sem_only = [c for c in sem_cols if c not in combined and 'Mean' not in c]

        if mean_only and sem_only:
            mc, sc, use_combined = mean_only[0], sem_only[0], False
        elif combined:
            mc, sc, use_combined = combined[0], None, True
        else:
            continue

        n_match = re.search(r'n=(\d+)', mc)
        n_orig = int(n_match.group(1)) if n_match else 6

        rows = []
        for _, row in df.iterrows():
            fa = str(row[fa_col]).strip()
            is_derived = fa in summary_names

            if use_combined:
                val_str = str(row[mc]).strip()
                if val_str.upper() == 'ND':
                    continue
                mean, sem = parse_mean_sem_str(row[mc])
                if mean is None:
                    mean = to_float(row[mc])
                    sem = None
            else:
                raw_mean = row[mc]
                if str(raw_mean).strip().upper() == 'ND':
                    continue
                mean = to_float(raw_mean)
                sem = to_float(row[sc])

            if mean is None or (isinstance(mean, float) and np.isnan(mean)):
                continue
            if sem is None or (isinstance(sem, float) and np.isnan(sem)):
                sem = 0.0

            sd = sem * np.sqrt(n_orig) if sem > 0 else max(mean * 0.05, 0.001)

            rows.append({
                'feature': fa, 'mean': mean, 'sem': sem,
                'sd': max(sd, 0.001),
                'min': max(0, mean - 3 * sd),
                'max': mean + 3 * sd,
                'n': n_orig, 'is_derived': is_derived
            })

        if rows:
            result[group] = pd.DataFrame(rows)
    return result


# ══════════════════════════════════════════════════════════════════════════
# 3. CORRELATIONS (LITERATURE)
# ══════════════════════════════════════════════════════════════════════════

def normalize_name(s):
    return re.sub(r'[^a-z0-9:]', '', s.lower())


def get_correlations():
    C = {}
    C[1] = {
        ('bodymass', 'totallength'): 0.95,
        ('bodymass', 'standardlength'): 0.94,
        ('bodymass', 'headlength'): 0.90,
        ('bodymass', 'bodymaximumheight'): 0.88,
        ('bodymass', 'bodymaximumcircumference'): 0.92,
        ('bodymass', 'bodymaximumthickness'): 0.83,
        ('totallength', 'standardlength'): 0.98,
        ('totallength', 'headlength'): 0.92,
        ('totallength', 'bodymaximumheight'): 0.85,
        ('totallength', 'bodymaximumcircumference'): 0.87,
        ('totallength', 'bodymaximumthickness'): 0.80,
        ('standardlength', 'headlength'): 0.90,
        ('standardlength', 'bodymaximumheight'): 0.84,
        ('standardlength', 'bodymaximumcircumference'): 0.86,
        ('standardlength', 'bodymaximumthickness'): 0.79,
        ('headlength', 'bodymaximumheight'): 0.78,
        ('headlength', 'bodymaximumcircumference'): 0.80,
        ('headlength', 'bodymaximumthickness'): 0.75,
        ('bodymaximumheight', 'bodymaximumcircumference'): 0.90,
        ('bodymaximumheight', 'bodymaximumthickness'): 0.88,
        ('bodymaximumcircumference', 'bodymaximumthickness'): 0.85,
    }
    C[2] = {
        ('profileindex', 'fultoncoefficient'): -0.60,
        ('profileindex', 'fleshyindex'): 0.40,
        ('fultoncoefficient', 'qualityindex'): 0.70,
        ('fultoncoefficient', 'thicknessindex'): 0.55,
        ('fultoncoefficient', 'fleshyindex'): 0.20,
        ('qualityindex', 'fleshyindex'): 0.30,
        ('qualityindex', 'thicknessindex'): 0.45,
        ('thicknessindex', 'fleshyindex'): 0.25,
        ('profileindex', 'qualityindex'): -0.35,
        ('profileindex', 'thicknessindex'): -0.30,
    }
    C[3] = {
        ('livemass', 'carcassmass'): 0.98,
        ('livemass', 'torsomass'): 0.95,
        ('livemass', 'filletmass'): 0.93,
        ('carcassmass', 'torsomass'): 0.96,
        ('carcassmass', 'filletmass'): 0.94,
        ('torsomass', 'filletmass'): 0.97,
        ('carcassyield', 'torsoyield'): 0.50,
        ('carcassyield', 'filletyield'): 0.55,
        ('torsoyield', 'filletyield'): 0.75,
        ('livemass', 'carcassyield'): 0.20,
        ('livemass', 'filletyield'): 0.25,
        ('carcassmass', 'carcassyield'): 0.30,
        ('filletmass', 'filletyield'): 0.40,
    }
    C[6] = {
        ('c14:0', 'c16:0'): 0.60, ('c16:0', 'c18:0'): 0.55,
        ('c14:0', 'c18:0'): 0.40, ('c16:0', 'c16:1'): 0.50,
        ('c18:0', 'c18:1'): -0.30, ('c16:1', 'c18:1'): 0.65,
        ('c18:2', 'c18:3'): 0.45, ('c18:2', 'c20:4'): 0.35,
        ('c18:3', 'c20:5'): 0.40, ('c20:5', 'c22:6'): 0.70,
        ('c20:5', 'c22:5'): 0.60, ('c16:0', 'c18:2'): -0.35,
        ('c16:0', 'c22:6'): -0.25, ('c18:1', 'c18:2'): -0.40,
        ('c18:1', 'c22:6'): -0.30,
    }
    return C


def get_storage_correlations():
    return {
        ('losses', 'water'): 0.80, ('proteins', 'lipids'): -0.35,
        ('proteins', 'water'): 0.30, ('lipids', 'water'): -0.50,
        ('ash', 'proteins'): 0.20, ('ash', 'lipids'): -0.15,
    }


# ══════════════════════════════════════════════════════════════════════════
# 4. COPULA ENGINE
# ══════════════════════════════════════════════════════════════════════════

def ensure_psd(R):
    if np.min(np.linalg.eigvalsh(R)) >= 1e-10:
        return R
    try:
        from statsmodels.stats.correlation_tools import corr_nearest
        return corr_nearest(R, threshold=1e-10)
    except ImportError:
        ev = np.maximum(np.linalg.eigvalsh(R), 1e-8)
        V = np.linalg.eigh(R)[1]
        R2 = V @ np.diag(ev) @ V.T
        d = np.sqrt(np.diag(R2))
        R2 /= np.outer(d, d)
        np.fill_diagonal(R2, 1.0)
        return R2


def build_corr(features, spec):
    n = len(features)
    R = np.eye(n)
    norm_idx = {normalize_name(f): i for i, f in enumerate(features)}
    for (a, b), r in spec.items():
        na, nb = normalize_name(a), normalize_name(b)
        ia = norm_idx.get(na)
        ib = norm_idx.get(nb)
        if ia is None:
            for nf, idx in norm_idx.items():
                if na in nf or nf in na:
                    ia = idx; break
        if ib is None:
            for nf, idx in norm_idx.items():
                if nb in nf or nf in nb:
                    ib = idx; break
        if ia is not None and ib is not None and ia != ib:
            R[ia, ib] = r; R[ib, ia] = r
    return ensure_psd(R)


def copula_generate(features, means, sds, mins, maxs, R, n_samples=1000, seed=42):
    rng = np.random.default_rng(seed)
    k = len(features)
    L = cholesky(ensure_psd(R), lower=True)
    Z = L @ rng.standard_normal((k, n_samples))
    U = norm.cdf(Z)
    X = np.zeros((n_samples, k))
    for i in range(k):
        if sds[i] < 1e-12:
            X[:, i] = means[i]; continue
        a = (mins[i] - means[i]) / sds[i]
        b = (maxs[i] - means[i]) / sds[i]
        if a >= b:
            X[:, i] = means[i]; continue
        X[:, i] = truncnorm.ppf(np.clip(U[i], 1e-10, 1-1e-10),
                                a, b, loc=means[i], scale=sds[i])
    return pd.DataFrame(X, columns=features)


# ══════════════════════════════════════════════════════════════════════════
# 5. DERIVED CALCULATIONS
# ══════════════════════════════════════════════════════════════════════════

def calc_yields(df):
    df = df.copy()
    lm_col = next((c for c in df.columns if 'live' in c.lower() and 'mass' in c.lower()), None)
    if not lm_col:
        return df
    lm = df[lm_col]
    for c in df.columns:
        cl = c.lower()
        if 'mass' in cl and 'live' not in cl:
            yc = c.replace('mass', 'yield').replace('Mass', 'Yield')
            if '(g)' in yc: yc = yc.replace('(g)', '(%)')
            df[yc] = (df[c] / lm) * 100
    return df


def calc_fa_indices(df):
    df = df.copy()
    cols = df.columns.tolist()
    sfa = [c for c in cols if re.match(r'^C\s*\d+:\s*0$', c.strip())]
    mufa = [c for c in cols if re.match(r'^C\s*\d+:\s*1', c.strip())]
    n3 = [c for c in cols if ('n-3' in c or 'n−3' in c) and c.strip().startswith('C')]
    n6 = [c for c in cols if ('n-6' in c or 'n−6' in c) and c.strip().startswith('C')]
    pufa = list(set(n3 + n6))

    if sfa: df['ΣSFA'] = df[sfa].sum(axis=1)
    if mufa: df['ΣMUFA'] = df[mufa].sum(axis=1)
    if pufa: df['ΣPUFA'] = df[pufa].sum(axis=1)
    if n3: df['n-3'] = df[n3].sum(axis=1)
    if n6: df['n-6'] = df[n6].sum(axis=1)

    if 'n-3' in df.columns and 'n-6' in df.columns:
        df['n-3/n-6'] = df['n-3'] / df['n-6'].replace(0, np.nan)
        df['n-6/n-3'] = df['n-6'] / df['n-3'].replace(0, np.nan)
    if 'ΣPUFA' in df.columns and 'ΣSFA' in df.columns:
        df['PUFA/SFA'] = df['ΣPUFA'] / df['ΣSFA'].replace(0, np.nan)
    if all(x in df.columns for x in ['ΣMUFA', 'ΣPUFA', 'ΣSFA']):
        df['USFA/SFA'] = (df['ΣMUFA'] + df['ΣPUFA']) / df['ΣSFA'].replace(0, np.nan)

    c14 = df.get('C14:0', df.get('C 14:0', 0))
    c16 = df.get('C16:0', df.get('C 16:0', 0))
    c18s = df.get('C18:0', df.get('C 18:0', 0))
    mu = df.get('ΣMUFA', 0); n3s = df.get('n-3', 0); n6s = df.get('n-6', 0)

    ai_d = mu + n6s + n3s
    if isinstance(ai_d, pd.Series):
        df['AI'] = (4*c14 + c16) / ai_d.replace(0, np.nan)
    n3n6 = (n3s / pd.Series(n6s).replace(0, np.nan)).fillna(0) if isinstance(n6s, pd.Series) else 0
    ti_d = 0.5*mu + 0.5*n6s + 3*n3s + n3n6
    if isinstance(ti_d, pd.Series):
        df['TI'] = (c14 + c16 + c18s) / ti_d.replace(0, np.nan)
    return df


# ══════════════════════════════════════════════════════════════════════════
# 6. MASTER PIPELINE
# ══════════════════════════════════════════════════════════════════════════

def generate_all_synthetic(df_dict, n_per_group=1000, seed=42, verbose=True):
    """
    Parameters
    ----------
    df_dict : {1: df1, 2: df2, ..., 6: df6}
    n_per_group : samples per group
    seed : reproducibility

    Returns
    -------
    {1: synthetic_df1, ..., 6: synthetic_df6}
    """
    rng = np.random.default_rng(seed)
    lit = get_correlations()
    stor = get_storage_correlations()
    out = {}

    for key, df in df_dict.items():
        if verbose:
            print(f"\n{'='*50}\nTable {key}: {df.shape}")

        col0 = str(df.columns[0]).lower()
        cols_low = ' '.join(str(c).lower() for c in df.columns)

        if 'fatty' in col0:
            ttype = 'C'
        elif 'storage' in col0:
            ttype = 'B'
        else:
            ttype = 'A'

        if ttype == 'A':
            parsed = parse_type_A(df)
            cs = lit.get(key, {})
            frames = []
            for grp, sdf in parsed.items():
                feats = sdf['feature'].tolist()
                R = build_corr(feats, cs)
                syn = copula_generate(feats, sdf['mean'].values, sdf['sd'].values,
                                      sdf['min'].values, sdf['max'].values,
                                      R, n_per_group, rng.integers(0, 2**31))
                syn['group'] = grp
                if key == 3:
                    syn = calc_yields(syn)
                frames.append(syn)
                if verbose:
                    print(f"  {grp}: {len(feats)} features × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

        elif ttype == 'B':
            parsed = parse_type_B(df)
            frames = []
            for grp, sdf in parsed.items():
                days = sorted(sdf['day'].unique())
                for day in days:
                    ddf = sdf[sdf['day'] == day]
                    feats = ddf['feature'].tolist()
                    bases = ddf['feature_base'].tolist()
                    cspec = {}
                    for (a, b), r in stor.items():
                        for i, bi in enumerate(bases):
                            for j, bj in enumerate(bases):
                                if a in bi.lower() and b in bj.lower() and i != j:
                                    cspec[(feats[i], feats[j])] = r
                    R = build_corr(feats, cspec)
                    syn = copula_generate(feats, ddf['mean'].values, ddf['sd'].values,
                                          ddf['min'].values, ddf['max'].values,
                                          R, n_per_group, rng.integers(0, 2**31))
                    syn['group'] = grp; syn['storage_day'] = day
                    syn.columns = [c.rsplit('_day', 1)[0] if '_day' in c else c
                                   for c in syn.columns]
                    frames.append(syn)
                if verbose:
                    print(f"  {grp}: {len(days)} timepoints × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

        elif ttype == 'C':
            parsed = parse_type_C(df)
            cs = lit.get(key, {})
            frames = []
            for grp, sdf in parsed.items():
                base = sdf[~sdf['is_derived']].copy()
                if base.empty:
                    continue
                feats = base['feature'].tolist()
                R = build_corr(feats, cs)
                syn = copula_generate(feats, base['mean'].values, base['sd'].values,
                                      base['min'].values, base['max'].values,
                                      R, n_per_group, rng.integers(0, 2**31))
                syn = calc_fa_indices(syn)
                syn['group'] = grp
                frames.append(syn)
                if verbose:
                    nb = len(feats)
                    nd = len(syn.columns) - nb - 1
                    print(f"  {grp}: {nb} base FAs + {nd} derived × {n_per_group} ✓")
            if frames:
                out[key] = pd.concat(frames, ignore_index=True)

    if verbose:
        print(f"\n{'='*50}\nDONE!")
        for k, v in out.items():
            print(f"  Table {k}: {v.shape}")
    return out


def validate(synthetic_dict, table_key):
    df = synthetic_dict.get(table_key)
    if df is None:
        print(f"Table {table_key} not found."); return
    for grp in df['group'].unique():
        g = df[df['group'] == grp]
        num = g.select_dtypes(include=[np.number]).drop(columns=['storage_day'], errors='ignore')
        print(f"\n── Table {table_key}, {grp} (n={len(g)}) ──")
        print(num.describe().round(4).to_string())

In [ ]:
"""
Merge synthetic_dict into two unified CSVs
==========================================

Usage:
    synthetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42)

    static_df  = merge_static(synthetic_dict)   # tables 1,2,3,6
    storage_df = merge_storage(synthetic_dict)   # tables 4,5

    static_df.to_csv('synthetic_static.csv', index=False)
    storage_df.to_csv('synthetic_storage.csv', index=False)
"""



def merge_static(synthetic_dict):
    """
    Tables 1,2,3,6 → one row per fish, all features side by side.
    Columns prefixed: bio_, idx_, cut_, fa_
    """
    prefixes = {1: 'bio', 2: 'idx', 3: 'cut', 6: 'fa'}
    frames = []

    for key in [1, 2, 3, 6]:
        if key not in synthetic_dict:
            continue
        df = synthetic_dict[key].copy()
        df['sample_id'] = df.groupby('group').cumcount()
        p = prefixes[key]
        rename = {c: f"{p}_{c}" for c in df.columns if c not in ('group', 'sample_id')}
        frames.append(df.rename(columns=rename))

    merged = frames[0]
    for df in frames[1:]:
        merged = merged.merge(df, on=['group', 'sample_id'], how='outer')

    merged = merged.drop(columns=['sample_id'])
    cols = ['group'] + [c for c in merged.columns if c != 'group']
    return merged[cols]


def merge_storage(synthetic_dict):
    """
    Tables 4,5 → pivot wide: each (measure, day) becomes a column.
    Columns: stor_Losses(%)_day0, ..., chem_Proteins(%)_day15, etc.
    """
    prefixes = {4: 'stor', 5: 'chem'}
    parts = []

    for key in [4, 5]:
        if key not in synthetic_dict:
            continue
        df = synthetic_dict[key].copy()
        day_col = 'storage_day' if 'storage_day' in df.columns else 'storage'
        measures = [c for c in df.columns if c not in ('group', day_col)]
        df['sample_id'] = df.groupby([day_col, 'group']).cumcount()
        p = prefixes[key]

        for mc in measures:
            piv = df.pivot_table(index=['group', 'sample_id'],
                                 columns=day_col, values=mc, aggfunc='first')
            piv.columns = [f"{p}_{mc}_day{int(d)}" for d in piv.columns]
            parts.append(piv)

    wide = pd.concat(parts, axis=1).reset_index()
    wide = wide.drop(columns=['sample_id'])
    cols = ['group'] + [c for c in wide.columns if c != 'group']
    return wide[cols]

In [ ]:
synthetic_dict = generate_all_synthetic(df_dict, n_per_group=1000, seed=42, verbose=True)


In [ ]:
static_df  = merge_static(synthetic_dict)    # tables 1, 2, 3, 6
storage_df = merge_storage(synthetic_dict)   # tables 4, 5

static_df.to_csv('synthetic_static.csv', index=False)
storage_df.to_csv('synthetic_storage.csv', index=False)

print(f"static_df: {static_df.shape},  storage_df: {storage_df.shape}")


In [ ]:
all_frames = []
for key, syn_df in syntetic_dict.items():
    df = syn_df.copy()
    df.insert(0, 'source_table', key)
    if 'storage_day' not in df.columns:
        df['storage_day'] = pd.NA
    all_frames.append(df)

merged = pd.concat(all_frames, ignore_index=True, sort=False)
merged.to_csv('synthetic_all_merged.csv', index=False)

# 6. Full ML Pipeline (Water Quality → Fish Composition)

In [ ]:
"""
Full ML Pipeline — Silurus glanis PhD
======================================
1. Generate v2 dataset (water↔fish correlations)
2. Merge with synthetic static + storage
3. Train models with Pipeline(StandardScaler → Model) — NO data leakage
4. Save results

Usage:
    python ml_full_pipeline.py
"""



# ══════════════════════════════════════════════════════════════════
# STEP 1: Generate v2 PHD dataset with biological correlations
# ══════════════════════════════════════════════════════════════════

def generate_phd_dataset(n_per_group=1000, seed=42):
    """
    Generate water quality → fish quality dataset with real biological
    correlations based on S. glanis literature.

    Relationships encoded:
        Temp  → Lipids (+)  : Hallier et al. (2007) Food Chem. 103:808
        O2    → Protein (+) : Huss (1995) FAO Fish. Tech. Paper 348
        Temp  → Bodymass (+): Simeanu et al. (2022) Agriculture 12:2144
        NH4+  → Protein (-) : ammonia toxicity, protein catabolism
        Feed  → Lipids (+)  : Jankowska et al. (2007) Eur. Food Res. Technol.
    """
    rng = np.random.default_rng(seed)

    def _gen(group, n):
        ag = group == 'AG'
        temp = np.clip(rng.normal(22 if ag else 12, 3, n), 8, 30)
        pH = np.clip(rng.normal(7.5 if ag else 7.3, 0.15, n), 6.8, 8.2)
        O2 = np.clip(rng.normal(6.5 if ag else 9.0, 1.2, n), 3.5, 12)
        cl = np.clip(rng.normal(95 if ag else 75, 8, n), 50, 120)
        no2 = np.clip(rng.normal(0.14 if ag else 0.08, 0.03, n), 0.01, 0.25)
        no3 = np.clip(rng.normal(1.8 if ag else 1.0, 0.4, n), 0.1, 3.0)
        nh4 = np.clip(rng.normal(0.22 if ag else 0.10, 0.06, n), 0.02, 0.40)
        po4 = np.clip(rng.normal(0.14 if ag else 0.06, 0.04, n), 0.001, 0.25)
        feed = np.ones(n) if ag else np.zeros(n)

        mass = np.clip(1500 + 12*temp + 15*O2 - 2.5*cl + 300*feed
                       + rng.normal(0, 120, n), 1200, 2400)
        prot = np.clip(15.0 + 0.35*O2 - 0.12*temp - 8.0*nh4 + 5.0*no2
                       + 0.5*feed + rng.normal(0, 1.5, n), 10, 25)
        lip = np.clip(1.5 + 0.06*temp + 0.8*feed - 0.08*O2 + 3.0*po4
                      - 1.5*no2 + rng.normal(0, 0.3, n), 1.0, 5.5)

        return pd.DataFrame({
            'Group': group, 'Water_Temp_C': temp.round(2),
            'Water_pH': pH.round(3), 'Water_O2_mgL': O2.round(3),
            'Chlorides_mgL': cl.round(2), 'Nitrites_mgL': no2.round(4),
            'Nitrates_mgL': no3.round(3), 'Ammonium_mgL': nh4.round(4),
            'Phosphates_mgL': po4.round(4), 'Feed_Type': feed.astype(int),
            'Bodymass_g': mass.round(2),
            'Protein_perc': prot.round(3), 'Lipids_perc': lip.round(3),
        })

    df = pd.concat([_gen('AG', n_per_group), _gen('RG', n_per_group)],
                   ignore_index=True)
    df.insert(0, 'Fish_ID', range(1, len(df) + 1))
    return df


# ══════════════════════════════════════════════════════════════════
# STEP 2: Merge datasets
# ══════════════════════════════════════════════════════════════════

def prepare_full_dataset(phd_df, static_df, storage_df):
    """Merge PHD + static + storage, encode group, handle nulls."""
    full = pd.concat([
        phd_df,
        static_df.drop(columns=['group'], errors='ignore'),
        storage_df.drop(columns=['group'], errors='ignore')
    ], axis=1)
    full = full.drop(columns=[c for c in full.columns
                               if full[c].isnull().mean() > 0.3])
    full = full.fillna(full.median(numeric_only=True))
    le = LabelEncoder()
    full['Group_enc'] = le.fit_transform(full['Group'])
    return full


# ══════════════════════════════════════════════════════════════════
# STEP 3: Model training with Pipeline (no leakage)
# ══════════════════════════════════════════════════════════════════

def _pipelines(task):
    """StandardScaler INSIDE Pipeline → fits only on train fold."""
    S = StandardScaler
    if task == 'reg':
        return {
            'Linear Regression': Pipeline([('s', S()), ('m', LinearRegression())]),
            'Ridge':             Pipeline([('s', S()), ('m', Ridge(alpha=1.0))]),
            'Lasso':             Pipeline([('s', S()), ('m', Lasso(alpha=0.01))]),
            'Random Forest':     Pipeline([('s', S()), ('m', RandomForestRegressor(
                n_estimators=150, max_depth=8, min_samples_leaf=5,
                random_state=42, n_jobs=-1))]),
            'XGBoost':           Pipeline([('s', S()), ('m', xgb.XGBRegressor(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                random_state=42, verbosity=0))]),
            'LightGBM':          Pipeline([('s', S()), ('m', lgb.LGBMRegressor(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                random_state=42, verbose=-1))]),
            'SVR':               Pipeline([('s', S()), ('m', SVR(kernel='rbf', C=10.0))]),
            'KNN':               Pipeline([('s', S()), ('m', KNeighborsRegressor(
                n_neighbors=7, weights='distance'))]),
        }
    else:
        return {
            'Logistic Regression': Pipeline([('s', S()), ('m', LogisticRegression(
                max_iter=2000, random_state=42))]),
            'Random Forest':       Pipeline([('s', S()), ('m', RandomForestClassifier(
                n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))]),
            'XGBoost':             Pipeline([('s', S()), ('m', xgb.XGBClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                random_state=42, verbosity=0, eval_metric='logloss'))]),
            'LightGBM':            Pipeline([('s', S()), ('m', lgb.LGBMClassifier(
                n_estimators=150, max_depth=4, learning_rate=0.05,
                random_state=42, verbose=-1))]),
            'SVM':                 Pipeline([('s', S()), ('m', SVC(
                kernel='rbf', C=10.0, random_state=42))]),
            'KNN':                 Pipeline([('s', S()), ('m', KNeighborsClassifier(
                n_neighbors=7, weights='distance'))]),
        }


def run_experiment(full_df, name, features, target, task='reg'):
    """Train all models, evaluate on holdout + 5-fold CV."""
    feats = [f for f in features if f in full_df.columns]
    X, y = full_df[feats].values, full_df[target].values

    stratify = full_df['Group_enc'] if task == 'cls' else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify)

    cv = (KFold(5, shuffle=True, random_state=42) if task == 'reg'
          else StratifiedKFold(5, shuffle=True, random_state=42))

    pipes = _pipelines(task)
    rows = []

    for mname, pipe in pipes.items():
        pipe.fit(X_tr, y_tr)
        yp = pipe.predict(X_te)
        scoring = 'r2' if task == 'reg' else 'accuracy'
        cvs = cross_val_score(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)

        if task == 'reg':
            rows.append({
                'experiment': name, 'model': mname,
                'R2_test': round(r2_score(y_te, yp), 4),
                'RMSE': round(np.sqrt(mean_squared_error(y_te, yp)), 4),
                'MAE': round(mean_absolute_error(y_te, yp), 4),
                'CV_R2_mean': round(cvs.mean(), 4),
                'CV_R2_std': round(cvs.std(), 4),
            })
        else:
            rows.append({
                'experiment': name, 'model': mname,
                'Accuracy': round(accuracy_score(y_te, yp), 4),
                'F1': round(f1_score(y_te, yp, average='weighted'), 4),
                'Precision': round(precision_score(y_te, yp, average='weighted'), 4),
                'Recall': round(recall_score(y_te, yp, average='weighted'), 4),
                'CV_Acc_mean': round(cvs.mean(), 4),
                'CV_Acc_std': round(cvs.std(), 4),
            })

# Feature importance (RF)
    rf = pipes['Random Forest']
    rf.fit(X, y)
    imp = pd.DataFrame({
        'experiment': name, 'feature': feats,
        'importance': rf.named_steps['m'].feature_importances_
    }).sort_values('importance', ascending=False).head(10)

    return pd.DataFrame(rows), imp


def run_all(full_df, verbose=True):
    """Run all 8 experiments."""
    w = ['Water_Temp_C', 'Water_pH', 'Water_O2_mgL', 'Chlorides_mgL',
         'Nitrites_mgL', 'Nitrates_mgL', 'Ammonium_mgL', 'Phosphates_mgL']
    fa = [c for c in full_df.columns if c.startswith('fa_')]
    allx = (w + ['Group_enc', 'Feed_Type'] +
            [c for c in full_df.columns
             if c.startswith(('bio_', 'idx_', 'cut_', 'fa_', 'stor_', 'chem_'))])

    exps = [
        ('R1_Water→Protein',  w + ['Group_enc', 'Feed_Type'], 'Protein_perc',  'reg'),
        ('R2_Water→Lipids',   w + ['Group_enc', 'Feed_Type'], 'Lipids_perc',   'reg'),
        ('R3_Water→Bodymass', w + ['Group_enc', 'Feed_Type'], 'Bodymass_g',    'reg'),
        ('R4_All→Protein',    allx,                           'Protein_perc',  'reg'),
        ('R5_All→Lipids',     allx,                           'Lipids_perc',   'reg'),
        ('C1_Water→Group',    w + ['Feed_Type'],              'Group_enc',     'cls'),
        ('C2_FA→Group',       fa,                             'Group_enc',     'cls'),
        ('C3_All→Group',      allx,                           'Group_enc',     'cls'),
    ]

    all_res, all_imp = [], []
    for name, feats, target, task in exps:
        if verbose:
            print(f"\n{'='*55}\n{name}")
        r, i = run_experiment(full_df, name, feats, target, task)
        all_res.append(r)
        all_imp.append(i)
        if verbose:
            for _, row in r.iterrows():
                if task == 'reg':
                    print(f"  {row['model']:25s} R²={row['R2_test']:7.4f}  "
                          f"RMSE={row['RMSE']:8.4f}  "
                          f"CV_R²={row['CV_R2_mean']:.4f}±{row['CV_R2_std']:.4f}")
                else:
                    print(f"  {row['model']:25s} Acc={row['Accuracy']:.4f}  "
                          f"F1={row['F1']:.4f}  "
                          f"CV={row['CV_Acc_mean']:.4f}±{row['CV_Acc_std']:.4f}")

    return (pd.concat(all_res, ignore_index=True),
            pd.concat(all_imp, ignore_index=True))


# ══════════════════════════════════════════════════════════════════
# STEP 4: Main
# ══════════════════════════════════════════════════════════════════


# Generate / load
phd = generate_phd_dataset(n_per_group=1000, seed=42)
static = pd.read_csv('synthetic_static.csv')
storage = pd.read_csv('synthetic_storage.csv')

# Merge
full = prepare_full_dataset(phd, static, storage)
print(f"Dataset: {full.shape}")

# Train
results, importance = run_all(full, verbose=True)

# Save
phd.to_csv('silurus_glanis_phd_dataset_v2.csv', index=False)
results.to_csv('ml_results_v2.csv', index=False)
importance.to_csv('feature_importance_v2.csv', index=False)

print(f"\n✓ results: {results.shape}")
print(f"✓ importance: {importance.shape}")

# 7. Transfer Learning: *Silurus glanis* → *Acipenser* spp.

In [ ]:
"""
Transfer Learning Validation: Silurus glanis → Acipenser spp.
==============================================================

Models trained on S. glanis synthetic water quality → flesh composition data
are applied to predict flesh lipid/protein content of three sturgeon species
using published water quality and proximate composition data.

Sturgeon test cases (peer-reviewed sources):
    1. Acipenser stellatus — Florescu Gune et al. (2021); Dorojan et al. (2020)
    2. Acipenser baerii — Lopez et al. (2020)
    3. Huso huso — Ghomi et al. (2013)
"""



def run_transfer_test(phd_df, static_df, storage_df):
    """
    Train on Silurus glanis, predict sturgeon, return results DataFrame.

    StandardScaler inside Pipeline — no data leakage.
    """

    # ── Prepare training data ──
    full = pd.concat([
        phd_df,
        static_df.drop(columns=['group'], errors='ignore'),
        storage_df.drop(columns=['group'], errors='ignore')
    ], axis=1)
    full = full.drop(columns=[c for c in full.columns if full[c].isnull().mean() > 0.3])
    full = full.fillna(full.median(numeric_only=True))
    le = LabelEncoder()
    full['Group_enc'] = le.fit_transform(full['Group'])

    features = [
        'Water_Temp_C', 'Water_pH', 'Water_O2_mgL', 'Chlorides_mgL',
        'Nitrites_mgL', 'Nitrates_mgL', 'Ammonium_mgL', 'Phosphates_mgL',
        'Group_enc', 'Feed_Type'
    ]

    X_train = full[features].values
    targets = {
        'Lipids (%)': full['Lipids_perc'].values,
        'Protein (%)': full['Protein_perc'].values,
        'Body mass (g)': full['Bodymass_g'].values,
    }

    S = StandardScaler
    model_defs = [
        ('Ridge', lambda: Pipeline([('s', S()), ('m', Ridge(alpha=1.0))])),
        ('Random Forest', lambda: Pipeline([('s', S()), ('m', RandomForestRegressor(
            n_estimators=150, max_depth=8, random_state=42, n_jobs=-1))])),
        ('LightGBM', lambda: Pipeline([('s', S()), ('m', lgb.LGBMRegressor(
            n_estimators=150, max_depth=4, learning_rate=0.05,
            random_state=42, verbose=-1))])),
    ]

    # Train
    models = {}
    for tname, y in targets.items():
        for mname, mfn in model_defs:
            m = mfn()
            m.fit(X_train, y)
            models[(tname, mname)] = m

    # ── Sturgeon test cases ──
    sturgeons = [
        {
            'name': 'Acipenser stellatus',
            'common': 'Sevruga sturgeon',
            'source': 'RAS, Galați, Romania',
            'water_ref': 'Florescu Gune et al. (2021)',
            'flesh_ref': 'Dorojan et al. (2020)',
            'Water_Temp_C': 25.0, 'Water_pH': 7.70, 'Water_O2_mgL': 6.60,
            'Chlorides_mgL': 90.0, 'Nitrites_mgL': 0.32,
            'Nitrates_mgL': 32.20, 'Ammonium_mgL': 0.43,
            'Phosphates_mgL': 0.10, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 1.32,
                'Protein (%)': 17.86,
                'Body mass (g)': 331.3,
            },
        },
        {
            'name': 'Acipenser baerii',
            'common': 'Siberian sturgeon',
            'source': 'Farm, Agroittica Lombarda, Italy',
            'water_ref': 'Estimated from facility reports',
            'flesh_ref': 'Lopez et al. (2020)',
            'Water_Temp_C': 18.0, 'Water_pH': 7.40, 'Water_O2_mgL': 8.50,
            'Chlorides_mgL': 80.0, 'Nitrites_mgL': 0.10,
            'Nitrates_mgL': 5.0, 'Ammonium_mgL': 0.15,
            'Phosphates_mgL': 0.08, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 5.60,
                'Protein (%)': 17.60,
                'Body mass (g)': 6500.0,
            },
        },
        {
            'name': 'Huso huso',
            'common': 'Beluga sturgeon',
            'source': 'Concrete tanks, Sari, Iran',
            'water_ref': 'Estimated from RAS standards',
            'flesh_ref': 'Ghomi et al. (2013)',
            'Water_Temp_C': 22.0, 'Water_pH': 7.50, 'Water_O2_mgL': 7.00,
            'Chlorides_mgL': 95.0, 'Nitrites_mgL': 0.15,
            'Nitrates_mgL': 2.0, 'Ammonium_mgL': 0.20,
            'Phosphates_mgL': 0.12, 'Group_enc': 0, 'Feed_Type': 1,
            'actual': {
                'Lipids (%)': 3.92,
                'Protein (%)': 14.73,
                'Body mass (g)': 15000.0,
            },
        },
    ]

    # ── Predict & compare ──
    rows = []
    for ex in sturgeons:
        X_test = np.array([[ex[f] for f in features]])
        for tname in targets:
            actual = ex['actual'][tname]
            for mname, _ in model_defs:
                pred = models[(tname, mname)].predict(X_test)[0]
                err = pred - actual
                pct = (err / actual) * 100 if actual != 0 else 0
                rows.append({
                    'Species': ex['name'],
                    'Common Name': ex['common'],
                    'Rearing System': ex['source'],
                    'Water Quality Ref': ex['water_ref'],
                    'Flesh Composition Ref': ex['flesh_ref'],
                    'Target': tname,
                    'Model': mname,
                    'Actual': round(actual, 2),
                    'Predicted': round(pred, 2),
                    'Absolute Error': round(abs(err), 2),
                    'Error (%)': round(pct, 1),
                })

    return pd.DataFrame(rows)

In [ ]:
# `phd` was defined inside the ML pipeline cell above
phd_df = phd


In [ ]:
results = run_transfer_test(phd_df, static_df, storage_df)
results.to_csv('transfer_learning_results.csv', index=False)
print(f"Transfer learning results: {results.shape}")
results.head()
